# Example 1.1.2 / 2: Defining a Set with =

Companion notebook for **1.1.2 Set Declaration vs. Definition** in the AMPL lab Documentation, section "Defining a Set with `=`". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Defining a Set with `=`

A defined set always follows the data it depends on: lowering the cost of coils puts them in
`CHEAP` at once.

> **Pitfall: data for a defined set**
> A set defined with `=` can't receive members from Python, from a data file or from a `let`
> command.

From Python, assigning members to a defined set raises a `RuntimeError`:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set SHIFTS = {"Day", "Night"};   # defined in the model: data can't change it

Writing model.mod


### amplpy: data.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# This step fails on purpose, as on the page; the error is printed so the notebook goes on
try:
    # data.py
    ampl.set["SHIFTS"] = ["Day"]
except Exception as error:
    print(f"{type(error).__name__}: {error}")

RuntimeError: file -
line 1
offset 29
SHIFTS was defined in the model.
context:  update data SHIFTS;data; set  >>> SHIFTS <<<  := 'Day' ; model;



### Pure AMPL: data.dat

Pure AMPL; AMPL rejects the data statement:

In [5]:
%%writefile data.dat
set SHIFTS := Day;

Writing data.dat


In [6]:
# This step fails on purpose, as on the page; the error is printed so the notebook goes on
try:
    ampl.eval(r"""
reset;
reset options;
model model.mod;
data data.dat;
""")
except Exception:
    pass  # AMPL has printed the error above

[ERROR] 
	data.dat
	line 1 offset 4
	SHIFTS was defined in the model.
	context:  set  >>> SHIFTS <<<  := Day;
